In [1]:
import subprocess
import json
from fitburst.analysis.stats import compute_test_f
import time
import os
import csv
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def execute_fitburst(fit_scattering=True):
    # 1. Ensure we are in the correct directory (where data is)
    # This prevents the pipeline from getting confused by long path strings
    pipeline = ".venv/lib/python3.13/site-packages/fitburst/pipelines/fitburst_pipeline.py"
    data_file = "scat_time.npz" 
    
    # --- FIT 1: Without Scattering ---
    # Pass ONLY the filename 'scat_time.npz'
    cmd_no_scatter = f"python {pipeline} {data_file} --ref_freq 400.1953125 --verbose --downsample_freq 16"
    
    print("Executing Fit 1...")
    subprocess.run(cmd_no_scatter, shell=True, check=True)
    
    # Based on your code logic: input 'scat_time.npz' + --outfile -> 'results_fitburst_scat_time.json'
    expected_f1 = "results_fitburst.json"
    
    if not os.path.exists(expected_f1):
        # Debugging: If file is still missing, list what was actually created
        print(f"Error: {expected_f1} not found. Found these files instead:")
        print([f for f in os.listdir('.') if 'json' in f])
        raise FileNotFoundError(f"Could not find {expected_f1}")

    with open(expected_f1, 'r') as f1:
        data_1 = json.load(f1)
        chi_square_no_scatter = data_1["fit_statistics"]["chisq_final"]
        width = data_1["fit_statistics"]["bestfit_parameters"]["burst_width"]

    if fit_scattering:
        # --- FIT 2: With Scattering ---
        # Note: We pass the JSON from Fit 1 as the solution
        cmd_scatter = (f"python {pipeline} {data_file} --fit scattering_timescale "
                       f"--scattering_timescale {width[0]} --ref_freq 400.1953125 "
                       f"--solution {expected_f1} --verbose --downsample_freq 16 --outfile")
        
        print("Executing Fit 2...")
        subprocess.run(cmd_scatter, shell=True, check=True)
        
        # When running with --outfile a second time, it might name it:
        # 'results_fitburst_scat_time_scat_time.json' OR just overwrite the first one.
        expected_f2 = "results_fitburst_scat_time_scat_time.json"
        if not os.path.exists(expected_f2):
            expected_f2 = "results_fitburst_scat_time.json"

        with open(expected_f2, 'r') as f2:
            data_2 = json.load(f2)
            chi_square_scatter = data_2["fit_statistics"]["chisq_final"]

        p_value = compute_test_f(
            chisq_1=chi_square_scatter,
            chisq_2=chi_square_no_scatter,
            num_fit_parameters_1=8,
            num_fit_parameters_2=7,
            num_observations_1=165888,
            num_observations_2=165888
        )
        return p_value

    return chi_square_no_scatter

In [4]:
subprocess.run("rm *.json *.png", shell=True)

CompletedProcess(args='rm *.json *.png', returncode=0)

In [5]:
p_value = execute_fitburst()
print("P-value: ", p_value)

Executing Fit 1...
INFO: no solution file found or provided; proceeding with fit...
INFO: there are 16384 frequencies and 162 time samples.
INFO: there are 16384 good frequencies...
INFO: input data cube is already dedispersed!
INFO: setting 'dm' entry to 0, now considered a dm-offset parameter...
INFO: initial guess for 1-component model:
    * amplitude: [0.5]
    * arrival_time: [0.07]
    * burst_width: [0.001]
    * dm: [0.0]
    * dm_index: [-2]
    * ref_freq: [400.1953125]
    * scattering_index: [-4]
    * scattering_timescale: [0.01]
    * spectral_index: [0]
    * spectral_running: [-4]
INFO: computing dedispersion-index matrix
INFO: initializing model
INFO: removing the following parameters: dm_index, scattering_index, scattering_timescale
INFO: new list of fit parameters: amplitude, arrival_time, burst_width, dm, spectral_index, spectral_running
0.00000  0.50000  0.07000   -4.00000  0.01000  0.00100 0.00000  -4.00000
0.00416  0.07394  0.07007   -4.00000  0.01000  0.00102 0

In [6]:
try:
    with open("results_fitburst_scat_time.json", 'r') as file:
        load_file = json.load(file)
        scattering_timescale = load_file["fit_statistics"]["bestfit_parameters"]["scattering_timescale"][0]
        uncertainties = load_file["fit_statistics"]["bestfit_uncertainties"]["scattering_timescale"][0]
    
    print("Scattering timescale: ", (scattering_timescale*1000) / 1.15, " ms")
    print("Scattering timescale uncertainty: ", uncertainties * 1000, "ms")

except FileNotFoundError as e:
    print(e)


Scattering timescale:  2.62446290385239  ms
Scattering timescale uncertainty:  0.31418632037528094 ms


In [ ]:
## Save results in csv file
try:
    with open("results_fitburst_scat_time.json", 'r') as file:
        load_file = json.load(file)
        scattering_timescale = load_file["fit_statistics"]["bestfit_parameters"]["scattering_timescale"][0]
        uncertainties = load_file["fit_statistics"]["bestfit_uncertainties"]["scattering_timescale"][0]
    
    print("Scattering timescale: ", (scattering_timescale*1000) / 1.15, " ms")
    print("Scattering timescale uncertainty: ", uncertainties * 1000, "ms")

    fitburst_data = {
        "p_vaue": p_value,
        "scat_time": (scattering_timescale*1000) / 1.15,
        "uncertainty": uncertainties * 1000
    }

except Exception:
    fitburst_data = {
        "p_vaue": np.nan,
        "scat_time": np.nan,
        "uncertainty": np.nan
    }

file_name = "fitburst_measurement.csv"

# Check if file exists
file_exists = os.path.isfile(file_name)

with open(file_name, mode='a', newline='') as csvfile:
    # Defining the fieldnames based on the keys in our dictionary
    writer = csv.DictWriter(csvfile, fieldnames=fitburst_data.keys())

    # Write header only if the file is new
    if not file_exists:
        writer.writeheader()
    
    writer.writerow(fitburst_data)

print(f"Prediction saved to {file_name}")

In [ ]:
burst = np.load("scat_time.npy")

In [ ]:
def get_analysis_number(filename="fluence_variations.csv"):
    with open(filename, 'r', newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        row_count = sum(1 for row in reader)
    return row_count-1

In [ ]:
get_analysis_number()

In [ ]:
fig, ax = plt.subplots(2, 1, height_ratios=[1, 3], figsize=(4, 4))
fig.subplots_adjust(hspace=0.03)
ax[0].plot(burst.mean(axis=0), color='black')
ax[0].set_facecolor("#ddddff")
ax[0].set_xticks([])
ax[0].set_yticks([])
for spine in ax[0].spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1)
ax[1].imshow(burst, aspect='auto', origin='lower')
ax[1].set_xlabel("Time [ms]")
ax[1].set_ylabel("Frequency [MHz]")
for spine in ax[1].spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1)
ax[1].set_yticks(np.linspace(0, 256, 9), np.linspace(400, 800, 9).astype(int))
ax[1].minorticks_on()
plt.savefig(f"fluence_comp_{get_analysis_number()}.jpg", bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
subprocess.run("mv *.jpg new_analysis_images/", shell=True)

In [ ]:
## Remove all previous analyses
subprocess.run("rm *.json *.png *.npy *.npz", shell=True)